# temporal_event_model v1 inspection

This notebook can instantiate the temporal model without a checkpoint, run one dummy inference, and show output shapes.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in Path.cwd().resolve().parents if (p / 'research').exists()), Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ROOT

In [ ]:
import torch
from torch import nn
from research.temporal_event_model.v1.config import ModelConfig, LossConfig
from research.temporal_event_model.v1.model import TemporalEventPredictor
from research.temporal_event_model.v1.losses import temporal_next_chunk_loss

class DummyEventEncoder(nn.Module):
    def __init__(self, embedding_dim=32):
        super().__init__()
        self.projection = nn.Linear(14 + 128 * 16, embedding_dim)
    def forward(self, header_uint8, events_uint8):
        x = torch.cat([header_uint8.float(), events_uint8.float().flatten(1)], dim=1) / 255.0
        return self.projection(x)

B, K, H, E = 1, 16, 1, 32
model = TemporalEventPredictor(
    event_encoder=DummyEventEncoder(E),
    config=ModelConfig(embedding_dim=E, temporal_d_model=64, temporal_layers=1, temporal_heads=4, decoder_layers=1),
    context_chunks=K,
    target_chunks=H,
)
context_header = torch.randint(0, 256, (B, K, 14), dtype=torch.uint8)
context_events = torch.randint(0, 256, (B, K, 128, 16), dtype=torch.uint8)
target_header = torch.randint(0, 256, (B, H, 14), dtype=torch.uint8)
target_events = torch.randint(0, 256, (B, H, 128, 16), dtype=torch.uint8)
with torch.no_grad():
    output = model(context_header, context_events)
    loss = temporal_next_chunk_loss(output, target_header, target_events, LossConfig(), detailed=True)
print('chunk_embeddings', tuple(output.chunk_embeddings.shape))
print('header_bit_logits', tuple(output.header_bit_logits.shape))
print('event_bit_logits', tuple(output.event_bit_logits.shape))
print(loss.metrics)

In [ ]:
try:
    from torchinfo import summary
    summary(model, input_data=(context_header, context_events), depth=4, col_names=('input_size', 'output_size', 'num_params'))
except Exception as exc:
    print(f'torchinfo summary unavailable: {exc!r}')